# Genetic Wake-Word Search

End-to-end pipeline: dataset generation → genetic hyperparameter search → multi-tier model training → ONNX export → benchmark.

## Configuration
Set env vars before running (e.g. Kaggle secrets), or edit the **Config cell** below.

| Variable | Default | Description |
|----------|---------|-------------|
| `WAKE_WORD` | `hey jarvis` | Wake word phrase |
| `OUTPUT_DIR` | `./ww_output` | Root output directory |
| `LANG_CODE` | `en` | Language for TTS synthesis |
| `N_POSITIVE` | `200` | Number of positive samples to generate |
| `ADVERSARIAL` | `true` | Include adversarial negatives |
| `DOWNLOAD_AUGMENT` | `false` | Download HuggingFace augmentation data |
| `POPULATION` | `12` | Genetic search population size |
| `GENERATIONS` | `5` | Number of generations |
| `EPOCHS_PER_TRIAL` | `3` | Epochs per genetic trial |
| `SEARCH_FULL` | `false` | Full search space (slower) |
| `TIERS_TO_TRAIN` | `micro,small,filterbank_small` | Comma-separated tiers for final training |
| `FINAL_EPOCHS` | `30` | Epochs for final model training |
| `EXPORT_ONNX` | `true` | Export ONNX models |
| `DEVICE` | `auto` | Device: auto, cpu, cuda, mps |
| `SEED` | `42` | Random seed |

In [ ]:
import os

WAKE_WORD         = os.environ.get("WAKE_WORD",         "hey jarvis")
OUTPUT_DIR        = os.environ.get("OUTPUT_DIR",        "./ww_output")
LANG              = os.environ.get("LANG_CODE",         "en")
N_POSITIVE        = int(os.environ.get("N_POSITIVE",    "200"))
ADVERSARIAL       = os.environ.get("ADVERSARIAL",       "true").lower() == "true"
DOWNLOAD_AUGMENT  = os.environ.get("DOWNLOAD_AUGMENT",  "false").lower() == "true"
# Genetic search
POPULATION        = int(os.environ.get("POPULATION",    "12"))
GENERATIONS       = int(os.environ.get("GENERATIONS",   "5"))
EPOCHS_PER_TRIAL  = int(os.environ.get("EPOCHS_PER_TRIAL", "3"))
SEARCH_FULL       = os.environ.get("SEARCH_FULL",       "false").lower() == "true"
# Final models: which tiers to train after search
TIERS_TO_TRAIN    = os.environ.get("TIERS_TO_TRAIN",    "micro,small,filterbank_small").split(",")
FINAL_EPOCHS      = int(os.environ.get("FINAL_EPOCHS",  "30"))
EXPORT_ONNX       = os.environ.get("EXPORT_ONNX",       "true").lower() == "true"
DEVICE            = os.environ.get("DEVICE",            "auto")
SEED              = int(os.environ.get("SEED",          "42"))

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "librosa", "onnx", "onnxruntime", "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "ovos-vad-plugin-silero", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

import os
_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)
print(f"Platform: {_platform} | Device: {DEVICE} | Wake word: {WAKE_WORD!r}")

In [ ]:
from pathlib import Path
from ww_trainer.quickstart import QuickstartConfig, _run_or_load_datagen

cfg_base = QuickstartConfig(
    wake_word=WAKE_WORD, output_dir=Path(OUTPUT_DIR),
    n_positive=N_POSITIVE, lang=LANG,
    adversarial=ADVERSARIAL, download_augmentation=DOWNLOAD_AUGMENT,
    reuse_dataset=True, seed=SEED,
)
datagen_result = _run_or_load_datagen(cfg_base)
n_train = sum(1 for _ in open(datagen_result.train_csv))
n_test  = sum(1 for _ in open(datagen_result.test_csv))
print(f"Train: {n_train} samples | Test: {n_test} samples")

In [ ]:
from ww_trainer.sweep import run_genetic_search

genetic_result = run_genetic_search(
    metadata_csv=str(datagen_result.train_csv),
    population_size=POPULATION,
    generations=GENERATIONS,
    epochs_per_trial=EPOCHS_PER_TRIAL,
    featurizer_type="mfcc",
    device=DEVICE,
    output_dir=str(Path(OUTPUT_DIR) / "genetic"),
    full=SEARCH_FULL,
)
best_hp  = genetic_result["best_config"]
best_f1  = genetic_result["best_score"]
print(f"Best search F1 : {best_f1:.4f}")
print(f"Best config    : {best_hp}")

In [ ]:
import matplotlib.pyplot as plt

history = genetic_result["history"]
gens  = [h["generation"] for h in history]
bests = [h["best"]       for h in history]
avgs  = [h["avg"]        for h in history]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gens, bests, "o-",  label="Best F1")
ax.plot(gens, avgs,  "s--", label="Avg F1",  alpha=0.7)
ax.set_xlabel("Generation"); ax.set_ylabel("F1 (search)")
ax.set_title(f"Genetic Search Evolution — {WAKE_WORD!r}")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "evolution.png", dpi=120)
plt.show()

In [ ]:
from ww_trainer.quickstart import _train_from_datagen_result

final_results = []
for tier_name in TIERS_TO_TRAIN:
    model_out = Path(OUTPUT_DIR) / f"model_{tier_name}"
    tier_cfg = QuickstartConfig(
        wake_word=WAKE_WORD, output_dir=model_out,
        tier=tier_name,
        epochs=FINAL_EPOCHS,
        batch_size=best_hp.get("batch_size", 16),
        lr=best_hp.get("lr", 5e-4),
        export_onnx=EXPORT_ONNX,
        device=DEVICE,
        download_augmentation=False,
        seed=SEED,
    )
    result = _train_from_datagen_result(tier_cfg, datagen_result)
    final_results.append({
        "tier": tier_name,
        "f1":   result.metrics.get("f1", 0.0),
        "onnx": result.best_onnx_path,
        "pt":   result.best_model_path,
    })
    print(f"  {tier_name:20s}  F1={result.metrics.get('f1', 0):.3f}")

In [ ]:
names  = [r["tier"] for r in final_results]
scores = [r["f1"]   for r in final_results]
colors = plt.cm.tab10.colors[:len(names)]

fig, ax = plt.subplots(figsize=(max(5, len(names)*1.8), 4))
bars = ax.bar(names, scores, color=colors)
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=10)
ax.set_ylim(0, 1.1); ax.set_ylabel("F1")
ax.set_title(f"Tier Comparison — {WAKE_WORD!r}")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "tier_comparison.png", dpi=120)
plt.show()

In [ ]:
from ww_trainer.benchmark import run_benchmark, plot_results, save_results
from IPython.display import Image, display

bench_dir = Path(OUTPUT_DIR) / "benchmark"
bench_dir.mkdir(parents=True, exist_ok=True)
bench_report = run_benchmark(
    device=DEVICE,
    output_dir=str(bench_dir),
)
save_results(bench_report, bench_dir)
plot_results(bench_report, bench_dir)

for png in sorted(bench_dir.glob("*.png")):
    display(Image(str(png)))

In [ ]:
from IPython.display import Markdown, display

rows = ["| Tier | F1 | ONNX |", "|------|-----|------|"] 
for r in final_results:
    onnx_mark = "✓" if r["onnx"] and Path(r["onnx"]).exists() else "—"
    rows.append(f"| {r['tier']} | {r['f1']:.3f} | {onnx_mark} |")
display(Markdown("\n".join(rows)))

print(f"\nAll outputs saved to: {Path(OUTPUT_DIR).resolve()}")

## Next Steps

- **Run headlessly**: set env vars (Kaggle secrets or `os.environ`) and re-run all cells
- **More tiers**: set `TIERS_TO_TRAIN=micro,small,medium,large` or any of the 12 tiers in `tiers.py`
- **Deeper search**: set `SEARCH_FULL=true`, increase `POPULATION` and `GENERATIONS`
- **Full dataset**: set `N_POSITIVE=500`, `DOWNLOAD_AUGMENT=true` for production-quality models

### Resources
- [Quickstart guide](../docs/quickstart.md)
- [Training docs](../docs/training.md)
- [Hardware guide](../docs/hardware_guide.md)
- [All tiers reference](../docs/classifiers.md)
- [Search strategies](../docs/search_strategies.md)